In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

# =============================
# 1. CONFIGURATION
# =============================

# Replace these placeholders with actual file paths
algorithm_files = {
    "Fixed-Time Controller": "../results/detector_output/detector-output-fixed.xml",
    "AMP Controller": "../results/detector_output/detector-output-acyclic.xml",
    "CAM-AMP Controller": "../results/detector_output/detector-output-cam-varaiya",
    "CMP Controller": "../results/detector_output/detector-output-cyclic.xml",
    "CAM-CMP Controller": "../results/detector_output/detector-output-cam-cyclic.xml",
}

# Select only detectors you want to include (Option B)
# If None → include all detectors found in XML
selected_detectors = None

# Smoothing window (in minutes)
SMOOTHING_WINDOW = 10

# Output folder
OUTPUT_DIR = "plots_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================
# 2. XML PARSER
# =============================

def parse_lanearea_xml(xml_path):
    """
    Parses SUMO laneAreaDetector XML output.
    Returns a pandas DataFrame with all intervals.
    """

    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows = []
    for interval in root.iter("interval"):
        data = interval.attrib
        row = {
            "time_start": float(data["begin"]),
            "time_end": float(data["end"]),
            "time_min": float(data["end"]) / 60.0,
            "detector": data["id"],
            "meanSpeed": float(data["meanSpeed"]),
            "meanTimeLoss": float(data["meanTimeLoss"]),
            "meanOccupancy": float(data["meanOccupancy"]),
            "meanMaxJamLengthInVehicles": float(data["meanMaxJamLengthInVehicles"]),
            "meanHaltingDuration": float(data["meanIntervalHaltingDuration"]),
            "nVehLeft": float(data["nVehLeft"]),
            "nVehEntered": float(data["nVehEntered"]),
            "nVehSeen": float(data["nVehSeen"]),
            "meanVehicleNumber": float(data["meanVehicleNumber"]),
            "jamLengthInVehiclesSum": float(data["jamLengthInVehiclesSum"]),
        }
        rows.append(row)

    #print(pd.DataFrame(rows));

    return pd.DataFrame(rows)

# =============================
# 3. CLEANING + AGGREGATION
# =============================

def clean_and_aggregate(df, selected_detectors=None):
    df = df.copy()

    # Filter detectors
    if selected_detectors is not None:
        df = df[df["detector"].isin(selected_detectors)]

    # Replace SUMO invalid values (-1) with NaN
    df.replace(-1.0, np.nan, inplace=True)

    # Compute mean network-wide metric across detectors (Option B)
    agg = df.groupby("time_start").agg({
        "meanSpeed": "mean",
        "meanTimeLoss": "mean",
        "meanOccupancy": "mean",
        "meanMaxJamLengthInVehicles": "mean",
        "meanHaltingDuration": "mean",
        "meanVehicleNumber": "mean",
        "nVehLeft": "mean",
        "jamLengthInVehiclesSum": "mean"
    })

    return agg

# =============================
# 4. SMOOTHING + UNCERTAINTY
# =============================

def smooth(df):
    return df.rolling(window=SMOOTHING_WINDOW, center=True, min_periods=1).mean()

# =============================
# 5. PLOTTING FUNCTIONS
# =============================

def plot_metric(all_data, metric, ylabel, filename):
    plt.figure(figsize=(12, 7))
    for algo, df in all_data.items():
        if metric not in df.columns:
            continue
        
        # Compute mean and standard deviation
        mean_val = df[metric].mean()
        std_val = df[metric].std()

        # Build label with statistics
        label_text = f"{algo} (μ={mean_val:.3f}, σ={std_val:.3f})"
        #label_text = f"{algo} (μ={mean_val:.3f})"

        # Plot using enhanced legend label
        plt.plot(df.index, df[metric], label=label_text, linewidth=2)


    plt.xlim(0, 3600)
    plt.xticks(np.arange(0, 3601, 600))
    plt.xlabel("Simulation Time (s)")
    plt.ylabel(ylabel)
    plt.title(f"Result of {ylabel} against Simulation Time(s)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.close()

# =============================
# 6. MAIN PIPELINE
# =============================

all_data_smoothed = {}

for algo, xml_path in algorithm_files.items():

    print(f"Processing: {algo}")

    if "PATH_" in xml_path:
        print(f"  !!! File path for {algo} not set. Skipping.")
        continue

    df = parse_lanearea_xml(xml_path)
    df = clean_and_aggregate(df, selected_detectors)
    df_smooth = smooth(df)
    all_data_smoothed[algo] = df_smooth

# =============================
# 7. PLOTS
# =============================

plot_metric(all_data_smoothed, "meanTimeLoss", "Mean Time Loss (s)", "mean_time_loss.png")
plot_metric(all_data_smoothed, "jamLengthInVehiclesSum", "Queue Length (m)", "queue_length.png")
plot_metric(all_data_smoothed, "meanOccupancy", "Mean Occupancy (%)", "occupancy.png")
plot_metric(all_data_smoothed, "meanSpeed", "Mean Speed (m/s)", "mean_speed.png")

print("\nAll plots saved in:", OUTPUT_DIR)

Processing: Fixed-Time Controller
Processing: AMP Controller
Processing: CAM-AMP Controller
Processing: CMP Controller
Processing: CAM-CMP Controller

All plots saved in: plots_output
